# Stage 3 — ESCO Occupation Classification (Batch API)
## Notebook 3.7 of 7 — Extract Results for Previously Missing Records

**Role in the pipeline:**
The final notebook in Stage 3. Reads the missing-record batch output files downloaded by notebook 3.6, parses the ESCO classification results, and merges them back into the main result pickles alongside the records that were successfully classified in the first pass.

After this step, each daily result pickle in `STAGE3_RESULT_PATH` should contain all vacancy records — both those classified in the first pass and those recovered via the second pass. Any remaining unclassified records (if the second pass also failed) are captured in `missing_after`.

**Position in Stage 3 sequence:**
1. 3.1 — Create batch input files
2. 3.2 — Submit batch jobs and monitor status
3. 3.3 — Extract classification results
4. 3.4 — Split missing and complete classifications
5. 3.5 — Create batch input files for missing records
6. 3.6 — Submit batch jobs for missing records
7. ▶ **3.7 — Extract results for missing records** ← *you are here*

**Prerequisites:**
- Notebook 3.6 completed — `missing_output_batch_status == 'complete'` for all applicable rows

## 3.3.1. Load packages and configuration

Loads standard libraries and the shared `general` config module.

In [ ]:
%load_ext autoreload
%autoreload 2
import sys
import os
sys.path.append("../code")
import general as g
g.clean_memory()

Imports Stage 2/3 processing modules and pandas.

In [ ]:
import stage2 as st2
import stage3 as st3
import pandas as pd

## 3.3.2. Load stage 3 logs

Reads the current tracker. The commented-out lines show how to initialise `complete_result_status` and `missing_after` columns for the first run of this notebook.

In [ ]:
process_df = pd.read_pickle(g.Config.STAGE3_PROCESS_PATH)
process_df.head()

## 3.3.3. Read output batches and merge results

Final merge loop: for each row where `missing_output_batch_status == 'complete'` and `complete_result_status != 'complete'`:

1. Parses the missing-record batch output JSONL via `extract_esco_codes()`.
2. Loads the missing pickle from `STAGE3_RESULT_PATH/missing/`, drops the old (empty) `esco_title`/`esco_code` columns, and merges in the newly extracted ESCO codes.
3. Concatenates the now-classified missing records with the previously complete records from the main result pickle.
4. Resets the index and saves the combined DataFrame back to the main result pickle path.
5. Counts any still-missing records (if the second pass also returned no classification) and stores the count in `missing_after`.
6. Sets `complete_result_status = 'complete'`.

Rows with `missing_input_batch_status == 'empty'` have no missing records and are skipped.

In [ ]:
for _, row in process_df.iterrows():
    if row["complete_result_status"] == "complete":
        continue
    if row["missing_input_batch_status"] == "empty":
        continue
    print("Working on: ", row["input_file"], "")

    if row["missing_output_batch_status"] == "complete":
        esco_df = st3.extract_esco_codes(row["missing_output_batch_path"])

        missing_path = os.path.join(g.Config.STAGE3_RESULT_PATH, "missing", row["input_file"] + ".pkl")
        missing_df = pd.read_pickle(missing_path)
        missing_df['id'] = missing_df['id'].astype(str)
        missing_df.drop(columns=['esco_title', 'esco_code'], inplace=True)
        missing_df = pd.merge(missing_df, esco_df, on='id', how='left')

        complete_df = pd.read_pickle(row["result_path"])
        result_df = pd.concat([complete_df, missing_df])
        result_df.reset_index(drop=True, inplace=True)
        result_df.to_pickle(row["result_path"])

        process_df.loc[_, "complete_result_status"] = "complete"

        ser = result_df["esco_title"]
        missing_mask = ser.isna() | (ser.astype(str).str.len() < 3)
        missing = int(missing_mask.sum())
        process_df.loc[_, "missing_after"] = missing

process_df.to_pickle(g.Config.STAGE3_PROCESS_PATH)

Displays the final tracker state. Verify that `complete_result_status == 'complete'` for all rows with `missing_input_batch_status != 'empty'`, and that `missing_after == 0` (or acceptably low) across all files.

In [ ]:
process_df

---
© 2026 Yurii Kleban, Britta Rude. All rights reserved.